<a href="https://colab.research.google.com/github/YukinoshitaSherry/CSCI572-Information_Retrieval_And_Web_Search_Engines/blob/master/scCode_gpt4o_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import scanpy as sc
import anndata
import numpy as np
import optuna
from torch_geometric.nn import GATConv
from sklearn.model_selection import KFold

# Load Dataset
adata = sc.read_h5ad(r"/content/kang_normalized_hvg.h5ad")


In [ ]:
# Preprocessing Function
def preprocess_data(adata):
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=2000)
    adata = adata[:, adata.var['highly_variable']]
    return adata

adata = preprocess_data(adata)
X = adata.X
timepoints = adata.obs['timepoint'].values


In [ ]:
# HVAE Model
class HVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=64):
        super(HVAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z


In [ ]:
# GNN Model
class GNN(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GATConv(in_channels, 128, heads=4, concat=True)
        self.conv2 = GATConv(128*4, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x



In [ ]:
# LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x, _ = self.lstm(x)
        x = self.fc(x[:, -1, :])
        return x



In [ ]:
# Training Pipeline
def train_model(X, timepoints, latent_dim=64, lr=1e-3, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Convert to PyTorch tensors
    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

    # Initialize models
    hvae = HVAE(X.shape[1], latent_dim).to(device)
    lstm = LSTMModel(latent_dim).to(device)

    optimizer = optim.Adam(list(hvae.parameters()) + list(lstm.parameters()), lr=lr)
    loss_fn = nn.MSELoss()

    # K-fold cross-validation
    kf = KFold(n_splits=5)

    for epoch in range(epochs):
        epoch_loss = 0
        for train_index, val_index in kf.split(X):
            X_train, X_val = X_tensor[train_index], X_tensor[val_index]

            optimizer.zero_grad()
            X_recon, z = hvae(X_train)
            pred = lstm(z.unsqueeze(1))
            loss = loss_fn(pred, torch.tensor(timepoints[train_index], dtype=torch.float32).to(device))
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / 5:.4f}")

train_model(X, timepoints)